# EEGEncoder Source-Only — Zhou2016 LOSO
Enable **GPU** and **Internet** in Kaggle, then run the cell below. Each MI trial is filtered at 8–30 Hz and uses the interval `[0.5, 5.0 s)`. Primary protocol: source Sessions 1–2 train, source Session 3 validates checkpoints, target Sessions 1–2 never enter training or adaptation, and target Session 3 is the held-out test set.

In [ ]:
# One-cell Kaggle: Zhou2016 LOSO targets S01-S04. Enable GPU + Internet.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/sota-baselines-bcic2a'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/eegencoder-zhou2016')
SUBJECT_IDS = list(range(1, 5))
MAX_EPOCHS = 100
BATCH_SIZE = 64  # EEGEncoder authors' batch size, as on BCI IV-2a/2b
USE_RA = False
RUN_TAG = 'source-only-no-ra-mi8_30-t0p5_5p0'
MNE_DATA = Path('/kaggle/working/mne_data')
RESULT_ARCHIVE = Path(f'/kaggle/working/eegencoder_zhou2016_loso_ep{MAX_EPOCHS}_{RUN_TAG}_results')
TRAIN_LOG = Path(f'/kaggle/working/eegencoder_zhou2016_loso_ep{MAX_EPOCHS}_{RUN_TAG}.log')

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] process is running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    # A previous run modifies the YAML config locally. Force the disposable
    # Kaggle clone back to the fetched branch so rerunning this cell is safe.
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
config_text = (REPO_PATH / 'configs/eegencoder.yaml').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'best_model_path' in pipeline_text
assert 'Pure source-only evaluation' in pipeline_text
assert 'domain_adaptation: false' in config_text and 'riemannian_alignment: false' in config_text
assert 'model: "EEGEncoderBaseline"' in config_text
assert 'low_cut: 8.0' in config_text and 'trial_duration: 4.5' in config_text

model_text = (REPO_PATH / 'models/eegencoder.py').read_text(encoding='utf-8')
assert 'class EEGEncoder(nn.Module)' in model_text
print("Verified: authors' EEGEncoder architecture, pure source-only protocol.")
print('Verified: pure source-only graph and best source-validation checkpoint selection.')

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

override = f"import yaml; from pathlib import Path; p=Path('configs/eegencoder.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']={SUBJECT_IDS!r}; c['max_epochs_loso_zhou2016']={MAX_EPOCHS}; pre=c['preprocessing']['zhou2016']; pre['low_cut']=8.0; pre['high_cut']=30.0; pre['start']=0.5; pre['trial_duration']=4.5; pre['batch_size']={BATCH_SIZE}; pre['riemannian_alignment']={USE_RA!r}; pre['interaug']=False; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])

MNE_DATA.mkdir(parents=True, exist_ok=True)

# Zhou2016 is hosted on figshare, which intermittently answers Kaggle with
# HTTP 403. MOABB skips its own download when data.zip is already in place,
# so fetch it here with retries, or copy it from an attached Kaggle dataset.
def prefetch_zhou2016(mne_data):
    import glob, shutil, urllib.request, zipfile
    base = Path(mne_data) / 'MNE-zhou-2016'
    if (base / 'subject_1').is_dir() or (base / 'data').is_dir():
        print('Zhou2016 already extracted at', base, flush=True)
        return
    base.mkdir(parents=True, exist_ok=True)
    target = base / 'data.zip'

    def valid(path):
        try:
            with zipfile.ZipFile(path) as zf:
                return any(name.endswith('S1_1A.cnt') for name in zf.namelist())
        except (zipfile.BadZipFile, OSError):
            return False

    if target.exists() and valid(target):
        print('Zhou2016 data.zip already present:', target, flush=True)
        return
    for candidate in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
        if valid(candidate):
            shutil.copyfile(candidate, target)
            print('Zhou2016 data.zip copied from Kaggle input:', candidate, flush=True)
            return
    urls = ['https://ndownloader.figshare.com/files/3662952',
            'https://api.figshare.com/v2/file/download/3662952']
    # A browser user agent can receive figshare's bot-check page instead of
    # the file, so try plain client agents first.
    agents = ['python-requests/2.34.2', 'curl/8.5.0',
              'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) '
              'Chrome/126.0 Safari/537.36']
    last_error = None
    for attempt in range(1, 7):
        for url in urls:
            agent = agents[(attempt - 1) % len(agents)]
            try:
                request = urllib.request.Request(url, headers={'User-Agent': agent})
                with urllib.request.urlopen(request, timeout=120) as response, open(target, 'wb') as out:
                    shutil.copyfileobj(response, out)
                if valid(target):
                    print(f'Zhou2016 data.zip downloaded ({target.stat().st_size / 1e6:.1f} MB) '
                          f'from {url} on attempt {attempt}.', flush=True)
                    return
                last_error = 'downloaded file is not the Zhou2016 archive'
            except Exception as exc:
                last_error = exc
            print(f'Zhou2016 download attempt {attempt} via {url} failed: {last_error}', flush=True)
        time.sleep(min(30 * attempt, 120))
    raise RuntimeError(
        'Could not obtain Zhou2016 from figshare (last error: ' + str(last_error) + '). '
        'Download https://ndownloader.figshare.com/files/3662952 (data.zip) in a browser, '
        'upload it as a Kaggle dataset, attach it via Add Input, and rerun.')


prefetch_zhou2016(MNE_DATA)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_ZHOU2016_PATH': str(MNE_DATA)})
print('===== EEGENCODER SOURCE-ONLY | ZHOU2016 LOSO S01-S04 | 3 CLASSES | 8–30 Hz | 0.5–5.0 s | 1125 SAMPLES =====', flush=True)
print(f'Options: epochs={MAX_EPOCHS} | batch={BATCH_SIZE} | Adam lr=0.001 label_smoothing=0.2 +L2 | filter=8–30 Hz | RA=False | DA=False | IM-TTA=False | InterAug=False', flush=True)
print('Primary protocol: source S1-S2=train, source S3=validation, target S1-S2=unused in training, target S3=held-out test; all target sessions=auxiliary test only', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'eegencoder', '--dataset', 'zhou2016', '--loso', '--gpu_id', '0', '--no_interaug'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Zhou2016 LOSO complete. Kaggle output:', archive, flush=True)
